# Mission 1: Locate and Retrieve the Dataset
```{admonition} Overview
:class: note

Welcome, Urban Data Investigator.

Your first mission is to locate a real dataset in ioerDATA, retrieve its metadata, and download all openly accessible files using the Dataverse API.

By completing this mission, you will learn how to:

- 🔍 search for a dataset using a title or DOI  
- 🧾 identify the dataset PID (Persistent Identifier)  
- 📦 retrieve metadata via API  
- 🔐 distinguish between public and restricted files  
- ⬇️ download openly accessible data  

🏅 **Badge to unlock:** Data Collector
```

```{admonition} Tips
:class: tip, dropdown

You will work with the replication package:

**Localized assessment of urban forest structures with 3D indicators**

This dataset is a complete research package. It includes documentation, code, and data used to study urban forest structures.

🧠 Think of this dataset as your **case file** — your job is to understand how to access and use it properly.
```

```{admonition} Note
:class: note, dropdown

In real research, data access must be:

- reproducible  
- transparent  
- well-documented  

Using an API allows you to:

- automate data retrieval  
- avoid manual download errors  
- ensure others can reproduce your workflow  

👉 You are not just downloading data — you are building a **reproducible workflow**.
```

```{admonition} Think 🤔
:class: important

If a dataset contains both public and restricted files:

👉 Should your code try to download everything?

**No.**

A responsible workflow must:

- clearly download only accessible files  
- document what could not be accessed  
- respect access restrictions  

🔍 This is a key principle of reproducible and ethical data use.
```

# 🔍 Step 1: Search for the Dataset

In [6]:
import requests

# Base URL of ioerDATA (Dataverse instance)
base_url = "https://data.fdz.ioer.de"

# API endpoint for searching datasets
search_url = f"{base_url}/api/search"

# Search query (dataset title)
query = "Localized assessment of urban forest structures with 3D indicators"

# Parameters for the API request
params = {
    "q": query,
    "type": "dataset",
    "per_page": 10
}

# Send request to API
r = requests.get(search_url, params=params, timeout=30)
r.raise_for_status()  # Raise error if request failed

# Extract search results
items = r.json().get("data", {}).get("items", [])

# Check if results exist
if not items:
    raise ValueError(f"No dataset found for query: {query}")

# Take the top result
top = items[0]

# Extract Persistent Identifier (PID / DOI)
persistent_id = top.get("global_id")

# Print results
print("Top match title:", top.get("name"))
print("Found PID:", persistent_id)

# Safety check
if not persistent_id:
    raise ValueError("Search result did not include a PID. Try refining the search query.")

Top match title: Replication package for: Localized assessment of urban forest structures with 3D indicators
Found PID: doi:10.71830/CDAXYF


```{admonition} Tips
:class: tip

You have successfully located the dataset.

Check:

- Does the title match your expectation?  
- Was a PID found?  
- Could someone else use this PID to retrieve the same dataset?  

👉 If yes, your workflow is reproducible.
```

# 📦 Step 2: Retrieve Dataset Metadata

In [7]:
import json

# API endpoint for retrieving dataset metadata using PID
dataset_url = f"{base_url}/api/datasets/:persistentId/"

# Request metadata
r = requests.get(dataset_url, params={"persistentId": persistent_id}, timeout=30)
r.raise_for_status()

# Store metadata as Python dictionary
dataset_metadata = r.json()

# Save metadata locally (important for reproducibility)
with open("g_dataset_metadata.json", "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=2, ensure_ascii=False)

print("Saved full metadata to: g_dataset_metadata.json")
print("Metadata retrieved successfully.")

Saved full metadata to: g_dataset_metadata.json
Metadata retrieved successfully.


```{admonition} Explanation
:class: note

Metadata is your **evidence file**.

It tells you:

- what the dataset is about  
- who created it  
- which files are included  
- which files are public  
- which files are restricted  

👉 Before downloading anything, a researcher always checks metadata first.
```

# ⬇️ Step 3: Download Openly Accessible Files

In [5]:
import os
import requests

# Create folder to store downloaded files
output_folder = "data/g_raw"
os.makedirs(output_folder, exist_ok=True)

# Extract file list from metadata
files = (
    dataset_metadata.get("data", {})
    .get("latestVersion", {})
    .get("files", [])
)

# Safety check
if not files:
    raise ValueError("No files found in metadata.")

downloaded = []
skipped = []

# Loop through all files in dataset
for fe in files:
    # Check if file is restricted
    is_restricted = fe.get("restricted", True)

    # Extract file metadata
    df = fe.get("dataFile", {})
    file_id = df.get("id")
    filename = df.get("filename", f"file_{file_id}")
    declared_size = df.get("filesize")

    # Skip restricted files
    if is_restricted:
        skipped.append((filename, declared_size, "restricted=true in metadata"))
        continue

    # Skip if file ID missing
    if not file_id:
        skipped.append((filename, declared_size, "missing file_id"))
        continue

    # Construct download URL
    download_url = f"{base_url}/api/access/datafile/{file_id}"

    try:
        # Stream download (efficient for large files)
        with requests.get(download_url, stream=True, timeout=60) as r:
            r.raise_for_status()

            local_path = os.path.join(output_folder, filename)

            total_bytes = 0
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        total_bytes += len(chunk)

        downloaded.append((filename, total_bytes))

    except requests.exceptions.HTTPError:
        skipped.append((filename, declared_size, f"download blocked ({r.status_code})"))
    except requests.exceptions.RequestException as e:
        skipped.append((filename, declared_size, f"network error: {e}"))

# Print results
print("Downloaded files:")
for name, size in downloaded:
    print(f" - {name} ({size:,} bytes)")

print("\nSkipped files:")
for name, size, reason in skipped:
    size_str = f"{size:,} bytes" if isinstance(size, int) else "unknown size"
    print(f" - {name} ({size_str}) -> {reason}")

Downloaded files:
 - README.md (11,959 bytes)
 - lcz_2018_classcodes.csv (528 bytes)
 - localized_assessment_of_urban_forest_structures_with_3d_indicators_replication_static.html (13,596,982 bytes)
 - localized_assessment_of_urban_forest_structures_with_3d_indicators_replication_workflow.ipynb (99,742 bytes)
 - urban_forest_3d_indicators_graphical_abstract.png (202,373 bytes)

Skipped files:
 - amsterdam_3D_canopy_stats_lczv3_grid100m_lau2021_epsg28992.gpkg (6,025,216 bytes) -> restricted=true in metadata
 - amsterdam_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds (2,447,010 bytes) -> restricted=true in metadata
 - berlin_3D_canopy_stats_lczv3_grid100m_urau2021_epsg25833.gpkg (25,026,560 bytes) -> restricted=true in metadata
 - berlin_gwr_canopy_indicators_adaptive_bandwidth_12_bisquare_kernel.rds (13,396,251 bytes) -> restricted=true in metadata


```{admonition} Think 🤔
:class: important

Look at the output carefully.

Answer:

- Which files were downloaded?  
- Which files were skipped?  
- Why were they skipped?  
- What does this tell you about data accessibility?  

🧠 This is how researchers evaluate **data availability and limitations**.

```

```{admonition} 🎉 Mission Complete — Badge Unlocked!
:class: tip

## 🏆 Data Collector

You successfully completed your first mission!

You can now:

- locate datasets using a DOI or search query  
- retrieve metadata via the API  
- distinguish between public and restricted files  
- download openly accessible data in a reproducible way  

🧠 **You’ve taken your first step as an Urban Data Investigator.**

```

```{admonition} 🎮 Mission Progress
:class: note

**Your Journey**

🟢 **Mission 1: Locate & Retrieve Data** ← *Current*  
⚪ Mission 2: Inspect the Package 🔒  
⚪ Mission 3: Understand the Data 🔒  
⚪ Mission 4: Generate Insights 🔒  
⚪ Mission 5: Create Visual Evidence 🔒  
⚪ Mission 6: Map the City 🔒  
⚪ Mission 7: Extend the Analysis 🔒  
⚪ Final Mission: Tell the Data Story 🔒  

**Progress:** ▰▱▱▱▱▱▱▱ 1/8 Complete
```

```{admonition} 🚧 Next Mission — Coming Soon
:class: warning

## 🔍 Mission 2: Inspect the Replication Package

Your next task will take you deeper into the dataset.

You will learn how to:

- explore the structure of a replication package  
- understand what each file contains  
- identify which data is usable for analysis  
- interpret metadata like a researcher  

🧠 **Challenge Preview**

Can you already guess:

- Which file contains the main spatial data?  
- Which files are supporting documentation?  
- What might be missing due to access restrictions?  

🏁 **Status:** Locked  
🔓 Unlocks after completing this mission sequence  

Stay tuned — your investigation continues soon.
```